In [ ]:
!pip install --upgrade pip

In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu129


In [ ]:
pip install numba

In [ ]:
from numba import jit, cuda

In [ ]:
pip install langchain

In [ ]:
pip install accelerate

In [ ]:
pip install bitsandbytes

In [ ]:
from langchain.llms import HuggingFacePipeline
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModelForSeq2SeqLM

In [ ]:
from langchain import PromptTemplate, HuggingFaceHub, LLMChain

template = """Context: {context}, Question: {question}, Answer: """

prompt = PromptTemplate(template=template, input_variables=["context","question"])

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
print(torch.cuda.is_available())

In [ ]:
import pandas as pd

df = pd.read_csv('elaboration_data.csv',header=0)



In [ ]:
df.info()

In [ ]:
print(df.select_idx)

In [ ]:
human_question_df = df.loc[df['elaboration_type'] == 'human_question'][['context','question']]
human_question_df['context'] = human_question_df['context'].str.replace('\n', ' ', regex=True)
print(human_question_df.head())

In [ ]:
def generate_from_model(model, tokenizer):
  encoded_input = tokenizer(text, return_tensors='pt')
  output_sequences = model.generate(input_ids=encoded_input['input_ids'].cuda())
  return tokenizer.decode(output_sequences[0], skip_special_tokens=True)

In [ ]:
name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
pipe = pipeline("text-generation",model=name, model_kwargs= {"device_map": "auto"}, max_new_tokens=100, max_length=100)
local_llm = HuggingFacePipeline(pipeline=pipe)
llm_chain = LLMChain(prompt=prompt, 
                     llm=local_llm
                     )
human_question_elab_df = pd.DataFrame()

for index, row in human_question_df.iterrows():
    con = row['context']
    ques = row['question']
    elab = llm_chain.run(context = con, question= ques)
    row = {'context': con, 'question':ques , 'elab': elab}
    human_question_elab_df = pd.concat([human_question_elab_df, pd.DataFrame([row])], ignore_index=True)
    
human_question_elab_df.to_csv('DeepSeek-R1-Distill-Qwen-7B+QUD.csv',index=False)
torch.cuda.empty_cache()

In [ ]:
del pipe
torch.cuda.empty_cache()

In [ ]:
name = "SanketAI/FLAN-T5_instruct-mistral7b"
pipe = pipeline("text2text-generation",model=name, model_kwargs= {"device_map": "auto"}, max_new_tokens=100, max_length=100)
local_llm = HuggingFacePipeline(pipeline=pipe)
llm_chain = LLMChain(prompt=prompt, 
                     llm=local_llm
                     )
human_question_elab_df = pd.DataFrame()

for index, row in human_question_df.iterrows():
    con = row['context']
    ques = row['question']
    elab = llm_chain.run(context = con, question= ques)
    row = {'context': con, 'question':ques , 'elab': elab}
    human_question_elab_df = pd.concat([human_question_elab_df, pd.DataFrame([row])], ignore_index=True)
    
human_question_elab_df.to_csv('FLAN-T5_instruct-mistral7b+QUD.csv',index=False)
torch.cuda.empty_cache()

In [ ]:
del pipe
torch.cuda.empty_cache()